# 13.5 多模态生成 (Multimodal Generation)

> 🕐 预估学习时间：40分钟

除理解外，产业还要求文生图/文生视频/统一生成。常见路径：LLM 规划 + 扩散/流匹配解码器，或原生多模态自回归（Chameleon、Transfusion 等思路）。

本节涵盖：
- 文本到潜空间条件
- 简化扩散训练步
- LLM 作为布局/提示规划器
- 统一离散多模态自回归直觉
- 质量与安全护栏要点


## 1. 条件扩散：文本嵌入引导去噪

教学版：在 2D 潜空间做条件去噪，条件向量来自文本编码器。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)


class TextEncoder(nn.Module):
    def __init__(self, vocab=50, d=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)

    def forward(self, ids):
        return self.emb(ids).mean(1)


class Denoiser(nn.Module):
    def __init__(self, d_x=2, d_cond=32, hidden=64):
        super().__init__()
        self.t_emb = nn.Linear(1, hidden)
        self.net = nn.Sequential(
            nn.Linear(d_x + d_cond + hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, d_x),
        )

    def forward(self, x_t, t, cond):
        te = self.t_emb(t.unsqueeze(-1))
        inp = torch.cat([x_t, cond, te], dim=-1)
        return self.net(inp)


text_enc = TextEncoder()
denoiser = Denoiser()
opt = torch.optim.Adam(list(text_enc.parameters()) + list(denoiser.parameters()), lr=2e-3)

print('=== Conditional Diffusion (toy 2D) ===')
for step in range(80):
    # target latent depends on token-0 class
    ids = torch.randint(0, 50, (64, 4))
    y = torch.stack([ids[:, 0].float() / 50, (ids[:, 0].float() / 50) ** 2], dim=-1)
    y = (y - 0.5) * 2
    t = torch.rand(64)
    noise = torch.randn_like(y)
    x_t = (1 - t.unsqueeze(-1)) * y + t.unsqueeze(-1) * noise
    cond = text_enc(ids)
    pred = denoiser(x_t, t, cond)
    loss = F.mse_loss(pred, noise)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0 or step == 79:
        print(f'step={step:02d} noise_mse={loss.item():.4f}')
print(f'\nKey: Text-conditioned denoisers learn to remove noise guided by language embeddings.')


## 2. LLM 规划器：先结构化，再渲染

复杂图像/视频生成常两段式：LLM 产出场景图/镜头脚本/扩展提示，再交给扩散模型。便于控制与可编辑。


In [ ]:
def llm_plan(prompt: str) -> dict:
    # Toy planner: extract keywords as layout slots
    words = [w for w in prompt.lower().replace(',', ' ').split() if len(w) > 2]
    objects = words[:3] or ['object']
    return {
        'expanded_prompt': prompt + ', highly detailed, cinematic lighting',
        'layout': [{'object': o, 'box': [0.1 + 0.25 * i, 0.2, 0.3, 0.4]} for i, o in enumerate(objects)],
        ' Negatives': 'blurry, watermark, text artifacts',
    }


plan = llm_plan('a fox and a robot in a snowy forest')
print('=== LLM Planner ===')
for k, v in plan.items():
    print(f'{k}: {v}')
print(f'\nKey: Planning separates semantic control (LLM) from pixel synthesis (diffusion).')


## 3. 统一离散多模态自回归（直觉）

把图像 patch / 音频 codec token 与文本 token 放进同一词表，用下一个 token 预测统一训练。优点是一套目标；难点是词表设计、模态平衡与长序列代价。


In [ ]:
class UnifiedAR(nn.Module):
    def __init__(self, vocab=128, d=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        # modality id: 0=text, 1=image
        self.mod = nn.Embedding(2, d)
        self.block = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 128, batch_first=True), 2
        )
        self.head = nn.Linear(d, vocab)

    def forward(self, tok, mod):
        h = self.emb(tok) + self.mod(mod)
        T = tok.size(1)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        return self.head(self.block(h, mask=mask))


model = UnifiedAR()
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
print('=== Unified Multimodal AR ===')
for step in range(40):
    # first half text tokens, second half image tokens
    tok = torch.randint(0, 128, (16, 24))
    mod = torch.cat([torch.zeros(16, 12, dtype=torch.long), torch.ones(16, 12, dtype=torch.long)], dim=1)
    logits = model(tok[:, :-1], mod[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, 128), tok[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 39:
        print(f'step={step:02d} loss={loss.item():.4f}')
print(f'\nKey: One next-token objective can cover text and image tokens if modality embeddings align them.')


## 4. 生成质量与安全

- **质量**：美学评分、CLIPScore、人体偏好（PickScore）、视频时序一致性  
- **安全**：NSFW/暴露/侵权检测、提示拦截、输出水印  
- **可控性**：布局/姿态/身份保持（IP-Adapter 等）与拒绝无授权人脸


In [ ]:
def clip_score_proxy(text_emb, image_emb):
    text_emb = F.normalize(text_emb, dim=-1)
    image_emb = F.normalize(image_emb, dim=-1)
    return (text_emb * image_emb).sum(-1)


t = F.normalize(torch.randn(8, 32), dim=-1)
img_good = F.normalize(t + 0.1 * torch.randn(8, 32), dim=-1)
img_bad = F.normalize(torch.randn(8, 32), dim=-1)
print('=== Quality Proxy ===')
print('good align', clip_score_proxy(t, img_good).mean().item())
print('bad align', clip_score_proxy(t, img_bad).mean().item())
print(f'\nKey: Alignment scores are cheap filters; human/aesthetics models still needed for product QA.')


## 课后思考题

1. 两段式（LLM 规划 + 扩散）与原生统一 AR 在可控性/效率上如何取舍？
2. 视频生成相对图像多了哪些失败模式（时序、身份漂移）？
3. 如何防止生成模型被用于深度伪造与版权侵权？
4. 多模态词表中图像 token 占比过高会怎样影响纯文本能力？

---
> 本节涵盖了13.5 多模态生成的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
